<a href="https://colab.research.google.com/github/AhmedAlwaqidi/meeting_summariser_model/blob/chunks-summarizing/meeting_summariser_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Set-up working packages

In [1]:
!pip install -q faster-whisper transformers sentencepiece accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.6/39.6 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 51.7 MB/s eta 0:00:00


Upload the meeting audio

In [2]:
from google.colab import files

uploaded = files.upload()

audio_file = list(uploaded.keys())[0]

print("Uploaded:", audio_file)

Saving m_test.m4a to m_test.m4a
Uploaded: m_test.m4a


Transcribe using whisper model

In [3]:
from faster_whisper import WhisperModel

model = WhisperModel(
    "small",
    device="cuda",
    compute_type="float16"
)

segments, info = model.transcribe(
    audio_file,
    beam_size=5
)

transcript_segments = []

for segment in segments:
    transcript_segments.append({
        "start": segment.start,
        "end": segment.end,
        "text": segment.text.strip()
    })

print("Detected language:", info.language)
print("Language probability:", info.language_probability)

for segment in transcript_segments:
    print(

        f"[{segment['start']:.2f} - {segment['end']:.2f}] "
        f"{segment['text']}"
    )

Detected language: ar
Language probability: 0.43994140625
[0.00 - 12.00] مرحباً أحمد . أنا سأتحدث اليوم . أنا سأتحدث اليوم about the project that we have to do
[12.00 - 26.00] اليوم سوف نتحدث عن الأشياء التي ستكون أسائلت لكم . مصر أمير ومصر أيمان
[27.00 - 39.00] يمكنك أن تتحدث عن الأشياء ولكن يمكنك أن تتحدث عن الأشياء . هل هناك سؤال عن الأشياء ؟
[41.00 - 49.00] مرحباً أنا أيمان . لا أستطيع أن أتحدث عن الأشياء الآن لأن أنا مصر أمير
[49.00 - 63.00] لذا يمكنك أن تتحدث عن الأشياء . أمير أنا . حسناً لذا يمكنك أن تتحدث عن الأشياء . لذا أتحدث عن الأشياء ونفعلها
[63.00 - 68.00] مع الأشياء . بالتأكيد . لذا شكراً
[68.00 - 78.00] لذا هناك سؤال عن الأشياء . أنا مزي أحمد ومجدداً ومجدداً


Save the transcript

In [4]:
transcript = "\n".join(
    segment["text"]
    for segment in transcript_segments
)


#merge the transcripts with timestamps
for segment in transcript_segments:
    print(
        f"[{segment['start']:.2f}s - {segment['end']:.2f}s] "
        f"{segment['text']}"
    )

#same the transcript to a file
import json

with open("meeting_transcript.json", "w", encoding="utf-8") as f:
    json.dump(
        transcript_segments,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Transcript saved successfully.")

[0.00s - 12.00s] مرحباً أحمد . أنا سأتحدث اليوم . أنا سأتحدث اليوم about the project that we have to do
[12.00s - 26.00s] اليوم سوف نتحدث عن الأشياء التي ستكون أسائلت لكم . مصر أمير ومصر أيمان
[27.00s - 39.00s] يمكنك أن تتحدث عن الأشياء ولكن يمكنك أن تتحدث عن الأشياء . هل هناك سؤال عن الأشياء ؟
[41.00s - 49.00s] مرحباً أنا أيمان . لا أستطيع أن أتحدث عن الأشياء الآن لأن أنا مصر أمير
[49.00s - 63.00s] لذا يمكنك أن تتحدث عن الأشياء . أمير أنا . حسناً لذا يمكنك أن تتحدث عن الأشياء . لذا أتحدث عن الأشياء ونفعلها
[63.00s - 68.00s] مع الأشياء . بالتأكيد . لذا شكراً
[68.00s - 78.00s] لذا هناك سؤال عن الأشياء . أنا مزي أحمد ومجدداً ومجدداً
Transcript saved successfully.


Load model well do the summarization

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

llm = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("LLM loaded successfully.")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

LLM loaded successfully.


Split the transcript into chunks

In [6]:
def split_transcript(text, max_words=1200):
    words = text.split()
    chunks = []

    for i in range(0, len(words), max_words):
        chunk = " ".join(words[i:i + max_words])
        chunks.append(chunk)

    return chunks


transcript_chunks = split_transcript(transcript)

print("Number of chunks:", len(transcript_chunks))

for i, chunk in enumerate(transcript_chunks):
    print(f"Chunk {i + 1}: {len(chunk.split())} words")

Number of chunks: 1
Chunk 1: 106 words


Summarize each chunk

In [7]:
def generate_response(prompt, max_new_tokens=800):
    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(llm.device)

    outputs = llm.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.2,
        do_sample=True
    )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return response.strip()

Summarize all the chunks

In [8]:
chunk_summaries = []

for i, chunk in enumerate(transcript_chunks):
    print(f"Summarizing chunk {i + 1}/{len(transcript_chunks)}...")

    prompt = f"""
You are a professional meeting analysis assistant.

Summarize the following section of a meeting transcript.

Extract:

- Main topics discussed
- Important points
- Suggestions
- Decisions
- Action items
- People mentioned
- Deadlines mentioned

Do not invent information.
Only use information that appears in the transcript.

Write the summary in English.

Meeting section:

{chunk}
"""

    result = generate_response(
        prompt,
        max_new_tokens=800
    )

    chunk_summaries.append(result)

print("Finished summarizing all chunks.")

Summarizing chunk 1/1...
Finished summarizing all chunks.


In [9]:
combined_summaries = "\n\n".join(
    f"Section {i + 1}:\n{summary}"
    for i, summary in enumerate(chunk_summaries)
)


print(combined_summaries)

Section 1:
Summary:

Main Topics Discussed:
- Project discussion

Important Points:
- The meeting is about the project they need to work on.
- Ahmed will speak today and will ask questions.
- Amir and Aiman can discuss the topics but should focus on answering questions.
- There was a question about the topics, but no specific question was asked during the meeting.

Suggestions:
- No suggestions were made.

Decisions:
- No decisions were made.

Action Items:
- No action items were assigned.

People Mentioned:
- Ahmed
- Amir
- Aiman

Deadlines Mentioned:
- No deadlines were mentioned.


Final meeting summary

In [10]:
final_prompt = f"""
You are an expert meeting analysis assistant.

Using the meeting section summaries below, create one complete and
accurate meeting report.

Write everything in English.

Use exactly this structure:

# Meeting Summary

Give a concise overview of the entire meeting.

# Key Topics

List the main subjects discussed.

# Important Points

List the most important information from the meeting.

# Decisions

List decisions that were actually made.

# Suggestions

List important suggestions and who suggested them when known.

# Action Items

For every action item include:

- Task
- Responsible person, if mentioned
- Deadline, if mentioned

# Open Questions

List questions or issues that were discussed but not resolved.

IMPORTANT:
- Do not invent information.
- Do not assume that a suggestion became a decision.
- Do not assign tasks to people unless the transcript indicates it.
- If a responsible person or deadline is not specified, write "Not specified".
- Keep the report factual.

Meeting section summaries:

{combined_summaries}
"""

final_summary = generate_response(
    final_prompt,
    max_new_tokens=1500
)

print(final_summary)

# Meeting Summary

The meeting focused on discussing the ongoing project. Ahmed was scheduled to lead the discussion and would be asking questions. Amir and Aiman were tasked with addressing the questions and discussing the project topics. However, there was no specific question asked during the meeting, and no particular topic was highlighted for detailed discussion. 

# Key Topics

- Project discussion

# Important Points

- Ahmed will lead the discussion and ask questions.
- Amir and Aiman will discuss the project topics and answer questions.
- No specific question or topic was discussed during the meeting.
- No deadlines were mentioned.

# Decisions

- No decisions were made.

# Suggestions

- No suggestions were made.

# Action Items

- Not specified

# Open Questions

- No open questions were raised during the meeting.


Save the result

In [11]:
import json

meeting_data = {
    "language": "en",
    "transcript": transcript_segments,
    "chunk_summaries": chunk_summaries,
    "final_summary": final_summary
}

with open("meeting_analysis.json", "w", encoding="utf-8") as f:
    json.dump(
        meeting_data,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Meeting analysis saved successfully.")

Meeting analysis saved successfully.
